# House Prices Regression Portfolio Workflow

This notebook is the clean portfolio version of the original exploration. The detailed trial-and-error notebook is kept separately, while the reproducible modeling logic lives in `src/train.py`.

Goal: predict residential sale prices from structured property attributes and present the work as a business-facing valuation support project.

## Workflow

1. Load Kaggle `train.csv` and `test.csv`.
2. Apply domain-aware missing-value handling.
3. Engineer property size, age, quality, amenity, and neighborhood interaction features.
4. Train Lasso, Ridge, and XGBoost models on `log1p(SalePrice)`.
5. Blend model predictions and export a Kaggle submission file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np

from train import TARGET, ID_COL, build_feature_frames, build_models, run_cross_validation, train_and_predict

In [ ]:
train_df = pd.read_csv(PROJECT_ROOT / 'train.csv')
test_df = pd.read_csv(PROJECT_ROOT / 'test.csv')

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
feature_train, feature_test = build_feature_frames(train_df, test_df)
feature_cols = [col for col in feature_train.columns if col not in [TARGET, ID_COL]]

X = feature_train[feature_cols]
y = np.log1p(feature_train[TARGET])
X_test = feature_test[feature_cols]

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'str', 'category']).columns.tolist()

print('Training rows:', len(X))
print('Features before one-hot:', len(feature_cols))
print('Numeric features:', len(num_cols))
print('Categorical features:', len(cat_cols))

In [ ]:
models = build_models(num_cols, cat_cols)
cv_results = run_cross_validation(models, X, y, folds=5)
cv_results

In [ ]:
predictions = train_and_predict(models, X, y, X_test)
submission = pd.DataFrame({ID_COL: feature_test[ID_COL].astype(int), TARGET: predictions})
submission.to_csv(PROJECT_ROOT / 'submission_manual_tag_blend.csv', index=False)
submission.head()

## Key Takeaways

- Linear regularized models are strong on this small tabular dataset.
- XGBoost underperforms as a standalone model but adds useful diversity in a weighted blend.
- Location-quality interactions are more useful than using neighborhood alone.
- Repeated seed checks are important because single-split CV improvements can be very small.